# Diffusion LMS — Final Sound Map Testing

Use the **same sound maps, phone counts, layouts, and three-panel comparison format** as the supplied reference testing notebook. This notebook runs only **one Diffusion LMS model** (no alternative safety-buffer models).

**Seven maps:** Single Stationary, Multiple Stationary, Single Moving, Distributed Background Noise, Concert, Stadium, and F1 Race Track.

- **Basic maps:** `Basic_Sound_Map_Files/` — 5, 10, 20, 50, 100 phones.
- **Final venue maps:** `Working_Final_Maps_and_Files/` — 10, 30, 50, 100, 250 phones.
- **Layouts:** `random`, `grid`, `clustered` (seed 42).
- **Score:** RMSE and worst under-prediction in dB. By default, RMSE includes every valid evaluation-grid position, matching the supplied reference benchmark. Set `EXCLUDE_MEASURED=True` to optionally score only locations without microphones (using the venue `evaluation_mask`).
- **Plots:** Ground Truth / Diffusion Prediction / Relative Error, with the original venue geometry and neon-green microphone markers.

**Runtime:** F1 has 721 frames. `QUICK_CHECK=True` uses explicit frame subsampling for a practical first run. Set it to `False` for full-frame benchmarking and direct comparison with full-frame scores. Both modes use the same Diffusion parameters.

## 1. Imports, project paths and Diffusion LMS configuration

In [ ]:
import sys
import importlib
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Search the VS Code working directory and its parent directories.
PROJECT_ROOT = None
for candidate in (Path.cwd(), *list(Path.cwd().parents)[:4]):
    if ((candidate / "Basic_Sound_Map_Files").is_dir()
            and (candidate / "Working_Final_Maps_and_Files").is_dir()):
        PROJECT_ROOT = candidate.resolve()
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find Basic_Sound_Map_Files and Working_Final_Maps_and_Files. "
        "Open this notebook from the project root, or extract the supplied ZIP first."
    )

# The final map scripts use absolute imports such as `from final_single_wall import ...`.
for folder in (PROJECT_ROOT, PROJECT_ROOT / "Basic_Sound_Map_Files",
               PROJECT_ROOT / "Working_Final_Maps_and_Files"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))

from Basic_Sound_Map_Files.extract_sound import (
    prepare_interpolation_data as prepare_basic_data,
)
from Working_Final_Maps_and_Files.final_extract_sound_venues import (
    prepare_interpolation_data as prepare_venue_data,
)
from diffusion_acoustic_lms_revised import DiffusionAcousticLMS

# Keep the same model for every scene and configuration.
DIFFUSION_KWARGS = dict(
    n_basis=100,
    basis_width_ratio=0.08,
    learning_rate=0.15,
    n_iterations=900,
    communication_radius=None,
    warm_start=True,
    readout="mean",
)

BASIC_PHONE_COUNTS = [5, 10, 20, 50, 100]
VENUE_PHONE_COUNTS = [10, 30, 50, 100, 250]
LAYOUTS = ["random", "grid", "clustered"]
SEED = 42
PLOT_N_PHONES = 100
PLOT_LAYOUTS = ["grid", "random", "clustered"]
SHOW_MICROPHONE_GALLERY = True
# False matches the supplied reference RMSE calculation across all valid cells.
# True is an OPTIONAL held-out-location evaluation using venue evaluation_mask.
EXCLUDE_MEASURED = False

# Quick-check is deliberately NOT a full time-series benchmark.
# Set False to score EVERY frame, as in the full reference testing.
QUICK_CHECK = True
QUICK_STRIDES = {
    "single_moving": 20,        # 101 frames -> 6 frames
    "distributed_noise": 20,    # 100 frames -> 5 frames
    "stadium": 20,              # 91 frames -> 5 frames
    "f1": 90,                   # 721 frames -> 9 frames
}

ALL_RESULTS = []
PREDICTION_GALLERY = {}
SCENE_INPUTS = {}
print("Project root:", PROJECT_ROOT)
print("Diffusion settings:", DIFFUSION_KWARGS)
print("Mode:", "SUBSAMPLED QUICK CHECK" if QUICK_CHECK else "FULL-FRAME BENCHMARK")

## 2. Data preparation, evaluation and plotting functions

In [ ]:
SCENES = {
    "single_stationary": dict(title="Single Stationary Point Source", basic=True,
                              module="Basic_Sound_Map_Files.point_source"),
    "multiple_stationary": dict(title="Multiple Stationary Point Sources", basic=True,
                                module="Basic_Sound_Map_Files.multiple_point_sources"),
    "single_moving": dict(title="Single Moving Point Source", basic=True,
                          module="Basic_Sound_Map_Files.single_moving_point_source"),
    "distributed_noise": dict(title="Distributed Background Noise", basic=True,
                              module="Basic_Sound_Map_Files.distributed_background_noise"),
    "concert": dict(title="Concert Venue", basic=False,
                    module="Working_Final_Maps_and_Files.final_concert_map"),
    "stadium": dict(title="Ashton Gate Stadium", basic=False,
                    module="Working_Final_Maps_and_Files.final_stadium_map"),
    "f1": dict(title="F1 Race Track", basic=False,
               module="Working_Final_Maps_and_Files.final_race_track_map"),
}


def load_scene(scene_key):
    """Load the actual uploaded map; do not generate substitute sound fields."""
    spec = SCENES[scene_key]
    module = importlib.import_module(spec["module"])
    plt.close("all")   # basic scene files may create demo figures on import
    if spec["basic"]:
        sound_maps, x, y = module.sound_maps, module.x, module.y
        allowed_mask = np.ones((len(y), len(x)), dtype=bool)
    else:
        sound_maps, x, y, allowed_mask = module.load_sound_map()
    sound_maps = np.asarray(sound_maps)
    if sound_maps.ndim == 2:
        sound_maps = sound_maps[None, :, :]
    assert sound_maps.shape[1:] == (len(y), len(x))
    assert allowed_mask.shape == (len(y), len(x))
    return module, sound_maps, np.asarray(x), np.asarray(y), np.asarray(allowed_mask, bool)


def draw_scene_overlay(ax, scene_key, module, frame_index=0):
    """Draw the exact original geometry; basic maps do not have venue walls."""
    if scene_key == "concert":
        module.draw_venue(ax, treated=True)
    elif scene_key == "stadium":
        module.draw_venue(ax)
    elif scene_key == "f1":
        module.draw_venue(ax, show_cars=True, frame=frame_index)


def make_frame_indices(scene_key, n_frames):
    stride = QUICK_STRIDES.get(scene_key, 1) if QUICK_CHECK else 1
    indices = np.arange(0, n_frames, stride, dtype=int)
    return indices, stride


def score_prediction(sample, predicted):
    """Ground truth is for evaluation only and is never supplied to Diffusion LMS."""
    truth = np.atleast_2d(sample["true_levels"])
    predicted = np.atleast_2d(predicted)
    assert truth.shape == predicted.shape, (truth.shape, predicted.shape)
    mask = sample.get("evaluation_mask") if EXCLUDE_MEASURED else None
    if mask is None:
        mask = np.ones(truth.shape[1], dtype=bool)
    mask = np.asarray(mask, dtype=bool)
    error = predicted[:, mask] - truth[:, mask]
    if not error.size:
        raise ValueError("No evaluation points remain after masking microphone positions")
    rmse = float(np.sqrt(np.mean(error**2)))
    max_under = float(max(0.0, -np.min(error)))
    return rmse, max_under


def plot_comparison(scene_key, module, sample, predictions, x, y, allowed_mask,
                    original_frame_indices, n_phones, layout, original_frame=0):
    """Match the reference: Ground Truth / Prediction / Relative Error (%)."""
    original_frame_indices = np.asarray(original_frame_indices)
    frame = int(np.argmin(np.abs(original_frame_indices - original_frame)))
    actual_frame = int(original_frame_indices[frame])
    grid_shape = (len(y), len(x))
    mask = np.asarray(allowed_mask, dtype=bool)
    truth_map = np.full(grid_shape, np.nan)
    predicted_map = np.full(grid_shape, np.nan)
    truth_map[mask] = sample["true_levels"][frame]
    predicted_map[mask] = predictions[frame]
    relative_error = 100.0 * np.abs(predicted_map - truth_map) / np.maximum(np.abs(truth_map), 1e-12)
    vmin, vmax = float(np.nanmin(truth_map)), float(np.nanmax(truth_map))
    extent = [x.min(), x.max(), y.min(), y.max()]
    sound_cmap = plt.get_cmap("inferno").copy()
    sound_cmap.set_bad("#eeeeee")

    fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
    images = (
        axes[0].imshow(truth_map, origin="lower", extent=extent,
                       cmap=sound_cmap, vmin=vmin, vmax=vmax),
        axes[1].imshow(predicted_map, origin="lower", extent=extent,
                       cmap=sound_cmap, vmin=vmin, vmax=vmax),
        axes[2].imshow(relative_error, origin="lower", extent=extent,
                       cmap="magma", vmin=0),
    )
    axes[0].set_title(f"Ground Truth (t={actual_frame})")
    axes[1].set_title(f"Diffusion Prediction ({n_phones} phones, {layout})")
    axes[2].set_title("Relative Error (%)")
    for ax in axes:
        draw_scene_overlay(ax, scene_key, module, actual_frame)
        ax.set_xlabel("x (m)")
        ax.set_ylabel("y (m)")
        ax.set_aspect("equal")
    phones = sample["phone_positions"]
    for ax in axes[1:]:
        ax.scatter(phones[:, 0], phones[:, 1], c="#39FF14", s=24,
                   edgecolors="black", linewidths=0.55, zorder=15, label="Phones")
        ax.legend(loc="upper left", fontsize=8)
    for ax, im, label in zip(axes, images, ["dB SPL", "dB SPL", "Relative Error (%)"]):
        fig.colorbar(im, ax=ax, label=label, shrink=0.85)
    fig.suptitle(SCENES[scene_key]["title"] + " — Diffusion LMS", fontsize=13)
    plt.show()


def plot_phone_configurations(scene_key, module, sound_maps, x, y, allowed_mask,
                               prepared_data, original_frame_indices, frame=0):
    """The 5 x 3 placement gallery from the final microphone visualisation script."""
    idx = int(np.argmin(np.abs(np.asarray(original_frame_indices) - frame)))
    actual_frame = int(original_frame_indices[idx])
    plot_map = np.where(allowed_mask, sound_maps[idx], np.nan)
    vmin, vmax = float(np.nanmin(plot_map)), float(np.nanmax(plot_map))
    cmap = plt.get_cmap("inferno").copy()
    cmap.set_bad("#eeeeee")
    counts = BASIC_PHONE_COUNTS if SCENES[scene_key]["basic"] else VENUE_PHONE_COUNTS
    fig, axes = plt.subplots(len(counts), len(LAYOUTS), figsize=(16, 19), constrained_layout=True)
    for row, n in enumerate(counts):
        for col, layout in enumerate(LAYOUTS):
            ax = axes[row, col]
            im = ax.pcolormesh(x, y, plot_map, shading="auto", cmap=cmap,
                               vmin=vmin, vmax=vmax)
            draw_scene_overlay(ax, scene_key, module, actual_frame)
            phones = prepared_data[(n, layout)]["phone_positions"]
            ax.scatter(phones[:, 0], phones[:, 1], c="#39FF14", s=30,
                       edgecolors="black", linewidths=0.65, zorder=15)
            ax.set_title(f"{n} phones — {layout}")
            ax.set_aspect("equal")
            if row != len(counts) - 1:
                ax.set_xlabel("")
            if col != 0:
                ax.set_ylabel("")
    fig.colorbar(im, ax=axes, label="Sound Pressure Level (dB)", shrink=0.6)
    fig.suptitle(f"{SCENES[scene_key]['title']} — Microphone Configurations (frame {actual_frame})",
                 fontsize=16)
    plt.show()


def test_scene(scene_key):
    """One map: prepare all layouts, run 15 benchmarks, plot the reference format."""
    module, original_maps, x, y, allowed_mask = load_scene(scene_key)
    frame_indices, stride = make_frame_indices(scene_key, len(original_maps))
    sound_maps = original_maps[frame_indices]
    spec = SCENES[scene_key]
    counts = BASIC_PHONE_COUNTS if spec["basic"] else VENUE_PHONE_COUNTS
    if spec["basic"]:
        prepared_data = prepare_basic_data(sound_maps, x, y, counts, LAYOUTS, seed=SEED)
    else:
        prepared_data = prepare_venue_data(
            sound_maps, x, y, counts, LAYOUTS, seed=SEED, allowed_mask=allowed_mask
        )

    print(f"\n--- Benchmarking {spec['title']} ---")
    print(f"Frame count: {len(frame_indices)} / {len(original_maps)} | stride={stride}")
    if stride > 1:
        print("NOTE: subsampled-frame RMSE; do not compare directly with full-frame results")

    rows = []
    for count in counts:
        for layout in LAYOUTS:
            sample = prepared_data[(count, layout)]
            model = DiffusionAcousticLMS(**DIFFUSION_KWARGS)
            predicted = model.fit_predict(
                phone_positions=sample["phone_positions"],
                phone_levels=sample["phone_levels"],
                query_positions=sample["query_positions"], x=x, y=y,
            )
            rmse, max_under = score_prediction(sample, predicted)
            row = dict(scene=scene_key, count=count, layout=layout, rmse=rmse,
                       max_under_db=max_under, frames=len(frame_indices), stride=stride)
            rows.append(row)
            ALL_RESULTS.append(row)
            print(f"Phones: {count:3d} | Layout: {layout:10s} | "
                  f"RMSE: {rmse:.2f} dB | Worst Under-Prediction: {max_under:.2f} dB")
            if count == PLOT_N_PHONES and layout in PLOT_LAYOUTS:
                PREDICTION_GALLERY[(scene_key, layout)] = (sample, predicted.copy())

    preferred_frame = 80 if scene_key == "f1" else 0
    for layout in PLOT_LAYOUTS:
        sample, predicted = PREDICTION_GALLERY[(scene_key, layout)]
        plot_comparison(scene_key, module, sample, predicted, x, y, allowed_mask,
                        frame_indices, PLOT_N_PHONES, layout, preferred_frame)

    if SHOW_MICROPHONE_GALLERY:
        plot_phone_configurations(scene_key, module, sound_maps, x, y, allowed_mask,
                                  prepared_data, frame_indices, preferred_frame)
    SCENE_INPUTS[scene_key] = (len(original_maps), frame_indices.tolist())
    return pd.DataFrame(rows)

## 3. Single Stationary Point Source

Run the full **15 phone configurations** and plot Ground Truth / Diffusion Prediction / Relative Error with the same visual structure.

In [ ]:
single_stationary_results = test_scene("single_stationary")
display(single_stationary_results)

## 4. Multiple Stationary Point Sources

Run the full **15 phone configurations** and plot Ground Truth / Diffusion Prediction / Relative Error with the same visual structure.

In [ ]:
multiple_stationary_results = test_scene("multiple_stationary")
display(multiple_stationary_results)

## 5. Single Moving Point Source

Run the full **15 phone configurations** and plot Ground Truth / Diffusion Prediction / Relative Error with the same visual structure.

In [ ]:
single_moving_results = test_scene("single_moving")
display(single_moving_results)

## 6. Distributed Background Noise

Run the full **15 phone configurations** and plot Ground Truth / Diffusion Prediction / Relative Error with the same visual structure.

In [ ]:
distributed_noise_results = test_scene("distributed_noise")
display(distributed_noise_results)

## 7. Concert Venue

Run the full **15 phone configurations** and plot Ground Truth / Diffusion Prediction / Relative Error with the same visual structure.

In [ ]:
concert_results = test_scene("concert")
display(concert_results)

## 8. Ashton Gate Stadium

Run the full **15 phone configurations** and plot Ground Truth / Diffusion Prediction / Relative Error with the same visual structure.

In [ ]:
stadium_results = test_scene("stadium")
display(stadium_results)

## 9. F1 Race Track

Run the full **15 phone configurations** and plot Ground Truth / Diffusion Prediction / Relative Error with the same visual structure.

In [ ]:
f1_results = test_scene("f1")
display(f1_results)

## 10. Combined RMSE results and export

In [ ]:
all_results = pd.DataFrame(ALL_RESULTS)
display(all_results)

# Same sampling/layout comparisons across all maps.
for name in SCENES:
    subset = all_results[all_results["scene"] == name]
    if subset.empty:
        continue
    pivot = subset.pivot(index="count", columns="layout", values="rmse")
    print(f"\n{name} — RMSE (dB)")
    display(pivot)

output_csv = PROJECT_ROOT / "diffusion_final_testing_results.csv"
all_results.to_csv(output_csv, index=False)
print("Saved:", output_csv)
print("Frame strides:", {key: stride for key, stride in zip(all_results.scene, all_results.stride)})

## Interpretation and comparison notes

- **Same input and evaluation format:** 15 configurations per sound map, with the correct 2D mask in venues.
- **Unmeasured points:** by default the benchmark scores all valid map cells exactly as the original reference. Set `EXCLUDE_MEASURED=True` to additionally evaluate only unmeasured points in venues using the final extractor's `evaluation_mask`.
- **Time-frame matching:** full-frame RMSE can only be compared fairly with another model's RMSE evaluated on exactly the same frames. Use `QUICK_CHECK=False` for the complete trajectories, including all 721 F1 frames.
- **Relative Error (%):** this follows the reference notebook, but percentages computed from logarithmic dB numbers are *not* a physically meaningful relative energy error. Treat dB RMSE and dB under-prediction as primary metrics.
- **Walls and track:** `draw_venue()` adds authentic venue geometry to plots. **Diffusion LMS does not directly ingest wall or track geometry**; those features influence the simulated Ground Truth and valid microphone locations, not the Gaussian RBF model itself.
- **Model:** the included `diffusion_acoustic_lms_revised.py` is the same RBF + ATC LMS implementation used in the preceding Diffusion testing work. No buffer, peak boosting, or artificial output adjustment has been added.